# Laboratorio 8 – Máquinas Vectoriales de Soporte (SVM / SVR)

**Objetivo:** Implementar Support Vector Machines para clasificación multiclase (Económico / Intermedio / Caro) y Support Vector Regression para predicción continua
del precio, comparando ambos contra todos los modelos de Labs 4–7 bajo exactamente las mismas condiciones experimentales.

Se reutilizan exactamente: mismo preprocesamiento, mismo `train_test_split(random_state=42)`

## Configuración – Reproducción del Pipeline de Labs 4–7

In [1]:
import pyreadr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import (train_test_split, StratifiedKFold, GridSearchCV,
                                     cross_val_score)
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC, SVR, LinearSVC
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, f1_score, precision_score,
                             recall_score, mean_squared_error, mean_absolute_error,
                             r2_score)
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.linear_model import LogisticRegression, RidgeCV

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


In [2]:
# Pipeline idéntico a Labs 4–7
result  = pyreadr.read_r('listings.Rdata')
df_raw  = result[list(result.keys())[0]].copy()
df      = df_raw.copy()

#  Precio
if df['price'].dtype == object:
    df['price'] = (df['price'].str.replace(r'[\$,]', '', regex=True)
                               .str.strip().replace('', np.nan).astype(float))
q_high = df['price'].quantile(0.99)
df = df[(df['price'] > 0) & (df['price'] <= q_high)].copy()

# Eliminar columnas innecesarias
cols_drop = ['id','listing_url','scrape_id','last_scraped','source','name',
    'description','neighborhood_overview','picture_url','host_url',
    'host_thumbnail_url','host_picture_url','host_about','host_verifications',
    'amenities','calendar_updated','calendar_last_scraped','license',
    'bathrooms_text','minimum_minimum_nights','maximum_minimum_nights',
    'minimum_maximum_nights','maximum_maximum_nights',
    'minimum_nights_avg_ntm','maximum_nights_avg_ntm']
df = df.drop(columns=[c for c in cols_drop if c in df.columns])

# Eliminar columnas con >60% nulos
null_pct = df.isnull().mean()
df = df.drop(columns=null_pct[null_pct > 0.60].index.tolist())

# Feature engineering de fechas
if 'host_since' in df.columns:
    df['host_since'] = pd.to_datetime(df['host_since'], errors='coerce')
    df['host_years'] = ((pd.Timestamp('2024-01-01') - df['host_since']).dt.days / 365).round(1)
    df = df.drop(columns=['host_since'])
df = df.drop(columns=[c for c in ['first_review','last_review'] if c in df.columns],
             errors='ignore')

# Booleans → 0/1
for col in ['host_is_superhost','host_has_profile_pic','host_identity_verified',
            'has_availability','instant_bookable']:
    if col in df.columns:
        df[col] = df[col].map({'t':1,'f':0,True:1,False:0})

# Porcentajes
for col in ['host_response_rate','host_acceptance_rate']:
    if col in df.columns:
        df[col] = df[col].str.replace('%','',regex=False).str.strip()
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Encoding de variables categóricas
TARGET = 'price'
num_features = [c for c in df.select_dtypes(include='number').columns if c != TARGET]
cat_features  = [c for c in ['room_type','property_type','neighbourhood_cleansed',
                               'host_response_time'] if c in df.columns]
for col in cat_features:
    freq = df[col].value_counts(normalize=True)
    df[col] = df[col].replace(freq[freq < 0.01].index, 'Otro')

df_encoded = pd.get_dummies(df[num_features + cat_features + [TARGET]],
                             columns=cat_features, drop_first=True, dtype=int)
for col in df_encoded.columns:
    if df_encoded[col].isnull().any():
        df_encoded[col].fillna(df_encoded[col].median(), inplace=True)

# Variable respuesta categórica
p33 = df_encoded[TARGET].quantile(0.33)
p67 = df_encoded[TARGET].quantile(0.67)
df_encoded['price_category'] = df_encoded[TARGET].apply(
    lambda p: 'Económico' if p <= p33 else ('Intermedio' if p <= p67 else 'Caro'))

feature_cols = [c for c in df_encoded.columns if c not in [TARGET, 'price_category']]
X        = df_encoded[feature_cols]
y_price  = df_encoded[TARGET]

# Splits IDÉNTICOS a todos los labs anteriores
X_train, X_test, y_train_price, y_test_price = train_test_split(
    X, y_price, test_size=0.20, random_state=42)

le = LabelEncoder()
y_cat = le.fit_transform(df_encoded['price_category'])
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X, y_cat, test_size=0.20, random_state=42, stratify=y_cat)

print(f"Dataset final: {df_encoded.shape[0]:,} filas x {len(feature_cols)} features")
print(f"Train completo: {len(X_train):,}  |  Test completo: {len(X_test):,}")
print(f"Umbral P33 = ${p33:.0f}  |  Umbral P67 = ${p67:.0f}")
print(f"Clases: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print("Pipeline reproducido — idéntico a Labs 4-7.")

Dataset final: 75,531 filas x 75 features
Train completo: 60,424  |  Test completo: 15,107
Umbral P33 = $140  |  Umbral P67 = $267
Clases: {'Caro': 0, 'Económico': 1, 'Intermedio': 2}
Pipeline reproducido — idéntico a Labs 4-7.


In [3]:
# Escalado StandardScaler + muestra estratificada para SVM
# SVM maximiza el margen entre clases. El margen se mide como distancia euclidiana en el espacio de features. Sin escalar, variables con rangos grandes (ej. 'availability_365': 0–365) dominarían el hiperplano sobre variables pequeñas (ej. 'review_scores_rating': 3–5), sesagando artificialmente el modelo.
# StandardScaler transforma a media=0, std=1 → todas las variables contribuyen equitativamente al cálculo del margen óptimo.

scaler      = StandardScaler()
X_train_sc  = scaler.fit_transform(X_train)   # fit SOLO en train
X_test_sc   = scaler.transform(X_test)         # transform en test con misma escala

# Muestra estratificada de 8,000 obs para SVM
# Con 60K+ filas la complejidad O(n²)–O(n³) hace inviable SVM completo.
# Se mantiene el test set COMPLETO para una comparación válida e imparcial.
N_SAMPLE = 8_000
np.random.seed(42)
idx_sample = []
for cls in np.unique(y_train_c):
    idx_cls = np.where(y_train_c == cls)[0]
    n_cls   = int(N_SAMPLE * (len(idx_cls) / len(y_train_c)))
    idx_sample.extend(np.random.choice(idx_cls, n_cls, replace=False))
idx_sample = np.array(idx_sample)

X_train_svm  = X_train_sc[idx_sample]
y_train_svm  = y_train_c[idx_sample]

# Mismo índice para regresión (SVR)
y_train_svmr = np.array(y_train_price)[idx_sample]
X_train_svmr = X_train_sc[idx_sample]

print(f"Muestra SVM: {len(X_train_svm):,} obs (de {len(X_train):,} en train)")
print(f"Test set (completo): {len(X_test_sc):,} obs")
print()
print("Distribución de clases en muestra SVM:")
for cls, name in zip(le.transform(le.classes_), le.classes_):
    n = (y_train_svm == cls).sum()
    print(f"  {name}: {n:,} ({n/len(y_train_svm)*100:.1f}%)")
print()
print("Distribución de precio en muestra SVR:")
print(pd.Series(y_train_svmr).describe().round(2).to_string())


Muestra SVM: 7,999 obs (de 60,424 en train)
Test set (completo): 15,107 obs

Distribución de clases en muestra SVM:
  Caro: 2,627 (32.8%)
  Económico: 2,641 (33.0%)
  Intermedio: 2,731 (34.1%)

Distribución de precio en muestra SVR:
count     7999.00
mean       337.06
std        796.98
min          8.00
25%        118.50
50%        185.00
75%        315.00
max      20000.00
